# DANFront - First Military Survey (1KF) tiles

Takes the raw First Survey GeoTIFFs straight from the SharePoint folder (single files, no archive), places each sheet with
Arcanum's DVD georeference (`ekf_mk.xml`, `geoview_mk.ini`, per-sheet `.ini`), cuts map tiles and uploads them to the
Cloudflare R2 bucket (`danfront/tiles-ext/`, the layer "Historical Map Overlay (1KF - DANFront)").
Sheets that the Arcanum layer already shows are skipped, so only the missing regions are added.
Nothing large is stored on Drive: only the list of finished sheets.

**Before you start**: the three R2 secrets (`R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ENDPOINT`) with *Notebook access* on,
and a runtime with many CPU cores (L4).

**How to use**: run the cells from top to bottom. After a disconnect run cells 1 to 5 again, then cell 7: finished sheets are skipped.

## 1. Setup (about 1 minute)
Mounts Google Drive and installs the tools. Approve the Drive permission window.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
WORK  = '/content/drive/MyDrive/danfront_1kf'   # on Drive: only the list of finished sheets
LOCAL = '/content/work'                          # on the Colab disk: downloaded sheets and tiles (temporary)
os.makedirs(WORK, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

!pip -q install rasterio pyproj
!command -v 7z >/dev/null || apt-get -qq install -y p7zip-full >/dev/null
!command -v rclone >/dev/null || (curl -s https://rclone.org/install.sh | bash >/dev/null 2>&1)
!7z | sed -n 2p; rclone version | head -1; df -h /content | tail -1

## 2. Load the scripts
Nothing to change here.

In [ ]:
%%writefile /content/work/tiles_2kf_batch.py
"""
tiles_2kf_batch.py - cut XYZ tiles from many Second Military Survey (2KF) GeoTIFFs at once,
all sheets merged into one tile folder. Used by colab_2kf.ipynb, also runnable by hand.

Input: GeoTIFFs exported with qgis_batch_export.py (Cassini-Soldner, origin St. Stephen's
Cathedral Vienna). Each sheet is clipped automatically to its map frame:
the sheets are 9600 x 6400 Vienna Klafter (18206.24 x 12137.50 m) cells of a regular grid in
that Cassini system, with Vienna in the middle of a cell, i.e. cell (i, j) covers
  X in [(i-0.5)*W, (i+0.5)*W],  Y in [(j-0.5)*H, (j+0.5)*H].
(Checked on 32-50Cas: the nominal cell matches the drawn frame within ~25 m.)
A sheet whose image does not cover most of one cell is tiled unclipped and reported as
"irregular"; files without a CRS are skipped.

Usage:
  python3 tiles_2kf_batch.py <tiles_out_dir> <sheet1.tif> [<sheet2.tif> ...] [--zoom 6,14] [--workers 2]
                             [--bbox lonmin,latmin,lonmax,latmax]

Prints one JSON line per sheet (status, cell, tile bounds) and writes the list of tile files
it created or changed to <tiles_out_dir>/../changed_tiles.txt (relative paths, for
`rclone copy --files-from`). Tiles shared by neighbouring sheets are composited, with a
file lock so parallel workers don't clash.
"""
import argparse, fcntl, json, math, os, sys
from multiprocessing import Pool
import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.warp import reproject, transform as warp_transform, Resampling
from rasterio.windows import from_bounds
from PIL import Image

R = 20037508.342789244
KLAFTER = 1.89648384
CELL_W, CELL_H = 9600 * KLAFTER, 6400 * KLAFTER
CASSINI_VIENNA = "+proj=cass +lat_0=48.208772 +lon_0=16.374242 +a=6376130 +rf=310 +units=m +no_defs"


def best_cell(b):
    """Grid cell with the largest overlap with image bounds b; returns (i, j, cell_bounds, covered_fraction_of_cell)."""
    best = None
    ci, cj = round((b.left + b.right) / 2 / CELL_W), round((b.bottom + b.top) / 2 / CELL_H)
    for i in (ci - 1, ci, ci + 1):
        for j in (cj - 1, cj, cj + 1):
            x0, x1, y0, y1 = (i - .5) * CELL_W, (i + .5) * CELL_W, (j - .5) * CELL_H, (j + .5) * CELL_H
            ox = max(0, min(b.right, x1) - max(b.left, x0)); oy = max(0, min(b.top, y1) - max(b.bottom, y0))
            frac = ox * oy / (CELL_W * CELL_H)
            if best is None or frac > best[3]:
                best = (i, j, (x0, y0, x1, y1), frac)
    return best


def unpremultiply(dst):
    """Mosaic pixels are premultiplied (edge pixels were averaged with transparent ones):
    turn them back into straight colour + alpha, so sheet edges don't come out dark."""
    a = dst[3].astype(np.uint16)
    out = dst.copy()
    part = (a > 0) & (a < 255)
    if part.any():
        for c in range(3):
            out[c][part] = np.minimum(255, (dst[c][part].astype(np.uint16) * 255 + a[part] // 2) // a[part])
    return out


def composite_save(path, dst, lock_path):
    """Merge a tile into the tile already on disk. Neighbouring sheets each cover part of an
    edge pixel, so their coverages are ADDED (new sheet first, old fills what is left) instead
    of the usual 'over' blend, which would leave a half-transparent line along every sheet edge."""
    new = unpremultiply(dst).transpose(1, 2, 0)
    with open(lock_path, "w") as lf:
        fcntl.flock(lf, fcntl.LOCK_EX)
        if os.path.exists(path):
            try:
                old = np.asarray(Image.open(path).convert("RGBA"))
                an = new[..., 3:].astype(np.float32)
                ao = np.minimum(old[..., 3:].astype(np.float32), 255 - an)
                tot = an + ao
                rgb = (new[..., :3] * an + old[..., :3] * ao) / np.maximum(tot, 1)
                new = np.concatenate([np.rint(rgb), tot], axis=2).astype(np.uint8)
            except OSError:
                pass                      # damaged tile left by an interrupted run: replace it
        os.makedirs(os.path.dirname(path), exist_ok=True)
        tmp = path + ".tmp"               # write to a temp file and rename, so an interruption
        Image.fromarray(np.ascontiguousarray(new), "RGBA").save(tmp, format="PNG", compress_level=6)   # can never leave a half-written tile behind
        os.replace(tmp, path)


def process(job):
    tif, out_dir, zmin, zmax, bbox = job
    sheet = os.path.splitext(os.path.basename(tif))[0]
    res = dict(sheet=sheet, status="", tiles=[], bounds={})
    try:
        with rasterio.open(tif) as ds:
            if ds.crs is None:
                res["status"] = "skipped: no CRS (RAW sheet)"; return res
            cas = rasterio.crs.CRS.from_proj4(CASSINI_VIENNA)
            same = abs(ds.crs.to_dict().get("lat_0", 0) - 48.208772) < 1e-6 and ds.crs.to_dict().get("proj") == "cass"
            if not same:
                res["status"] = f"skipped: unexpected CRS {ds.crs.to_proj4()[:60]}"; return res
            i, j, (x0, y0, x1, y1), frac = best_cell(ds.bounds)
            res["cell"] = [i, j]; res["cell_coverage"] = round(frac, 3)
            if frac >= 0.9:
                win = from_bounds(x0, y0, x1, y1, ds.transform).round_offsets().round_lengths()
                win = win.intersection(rasterio.windows.Window(0, 0, ds.width, ds.height))
                res["status"] = "ok"
            else:
                win = rasterio.windows.Window(0, 0, ds.width, ds.height)
                res["status"] = "ok, irregular (not clipped)"
            data = ds.read([1, 2, 3], window=win)
            src_tf = ds.window_transform(win); src_crs = ds.crs
        rgba = np.zeros((4,) + data.shape[1:], np.uint8)
        rgba[:3] = data; rgba[3] = np.where(data.any(axis=0), 255, 0); del data
        H, W = rgba.shape[1:]
        n = 20
        edge = ([(W*k/n, 0) for k in range(n+1)] + [(W, H*k/n) for k in range(n+1)] + [(W*k/n, H) for k in range(n+1)] + [(0, H*k/n) for k in range(n+1)])
        xs, ys = zip(*[src_tf * p for p in edge])
        mx, my = warp_transform(src_crs, "EPSG:3857", xs, ys)
        minX, maxX, minY, maxY = min(mx), max(mx), min(my), max(my)
        if bbox:
            lon, lat = warp_transform("EPSG:3857", "EPSG:4326", [minX, maxX], [minY, maxY])
            if lon[1] < bbox[0] or lon[0] > bbox[2] or lat[1] < bbox[1] or lat[0] > bbox[3]:
                res["status"] = "skipped: outside bbox"; return res
        lock_path = os.path.join(os.path.dirname(os.path.abspath(out_dir)), ".tiles.lock")
        # One warp per zoom level into a mosaic aligned to the tile grid, then slice it into tiles.
        # The deepest zoom is warped from the sheet; each shallower zoom is averaged from the one above it.
        prev = prev_tf = None
        for z in range(zmax, zmin - 1, -1):
            size = 2 * R / 2 ** z
            xa, xb = int((minX + R) // size), int((maxX + R) // size)
            ya, yb = int((R - maxY) // size), int((R - minY) // size)
            res["bounds"][z] = dict(xMin=xa, xMax=xb, yMin=ya, yMax=yb)
            mos = np.zeros((4, (yb - ya + 1) * 256, (xb - xa + 1) * 256), np.uint8)
            mtf = Affine(size / 256, 0, -R + xa * size, 0, -size / 256, R - ya * size)
            if prev is None:
                reproject(rgba, mos, src_transform=src_tf, src_crs=src_crs, dst_transform=mtf, dst_crs="EPSG:3857",
                          resampling=Resampling.bilinear)
            else:
                reproject(prev, mos, src_transform=prev_tf, src_crs="EPSG:3857", dst_transform=mtf, dst_crs="EPSG:3857",
                          resampling=Resampling.average)
            prev, prev_tf = mos, mtf
            for x in range(xa, xb + 1):
                for y in range(ya, yb + 1):
                    dst = mos[:, (y - ya) * 256:(y - ya + 1) * 256, (x - xa) * 256:(x - xa + 1) * 256]
                    if dst[3].max() == 0:
                        continue
                    rel = f"{z}/{x}/{y}.png"
                    composite_save(os.path.join(out_dir, rel), np.ascontiguousarray(dst), lock_path)
                    res["tiles"].append(rel)
    except Exception as e:
        res["status"] = f"ERROR: {type(e).__name__}: {e}"
    return res


def run(tifs, out_dir, zmin=6, zmax=14, workers=2, bbox=None):
    """Process the given GeoTIFFs; returns the list of per-sheet result dicts."""
    os.makedirs(out_dir, exist_ok=True)
    jobs = [(t, out_dir, zmin, zmax, bbox) for t in tifs]
    if workers > 1 and len(jobs) > 1:
        with Pool(workers) as p:
            results = p.map(process, jobs, chunksize=1)
    else:
        results = [process(j) for j in jobs]
    changed = sorted({t for r in results for t in r["tiles"]})
    with open(os.path.join(os.path.dirname(os.path.abspath(out_dir)), "changed_tiles.txt"), "w") as f:
        f.write("\n".join(changed) + ("\n" if changed else ""))
    return results


def merge_bounds(total, b):
    for z, v in b.items():
        z = int(z); t = total.setdefault(z, dict(v))
        t.update(xMin=min(t["xMin"], v["xMin"]), xMax=max(t["xMax"], v["xMax"]), yMin=min(t["yMin"], v["yMin"]), yMax=max(t["yMax"], v["yMax"]))
    return total


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("out_dir"); ap.add_argument("tifs", nargs="+")
    ap.add_argument("--zoom", default="6,14"); ap.add_argument("--workers", type=int, default=2); ap.add_argument("--bbox")
    a = ap.parse_args()
    zmin, zmax = map(int, a.zoom.split(","))
    bbox = tuple(map(float, a.bbox.split(","))) if a.bbox else None
    total = {}
    for r in run(a.tifs, a.out_dir, zmin, zmax, a.workers, bbox):
        merge_bounds(total, r["bounds"])
        print(json.dumps({k: (len(v) if k == "tiles" else v) for k, v in r.items() if k != "bounds"}, ensure_ascii=False))
    print("tile bounds:", json.dumps(total))


In [ ]:
%%writefile /content/work/tiles_1kf_batch.py
"""
tiles_1kf_batch.py - cut XYZ tiles from many First Military Survey (1KF) sheets at once, all
sheets merged into one tile folder. Same interface as tiles_2kf_batch.py (used by colab_1kf.ipynb).

The First Survey GeoTIFFs carry no coordinates ("RAW"). Each sheet is placed with Arcanum's own
DVD georeference (see tiles_1kf_dvd.py for the details):
  ekf_mk.xml      origin and cell size of every sheet in the DVD's grid system
  geoview_mk.ini  2nd-order polynomial from that grid system to UTM zone 33
  <sheet>.ini     clip=x,y,w,h : the map frame (margins are cut off)
Accuracy: a few hundred metres on average, up to ~2 km in the far corners of the country.

Call configure() once before run():
  configure(outlay_dir, ini_dir, covered_url=None, skip=())
    outlay_dir   folder with ekf_mk.xml and geoview_mk.ini
    ini_dir      folder with the per-sheet .ini files (searched recursively)
    covered_url  optional tile URL template of a layer that already shows this survey, e.g.
                 "https://.../tiles/{z}/{x}/{y}.png". A sheet is skipped when that layer already
                 has tiles at the sheet's centre and at its four quarter points.
    skip         sheet names to leave out (sheets already georeferenced with control points)

Usage by hand:
  python3 tiles_1kf_batch.py <outlay_dir> <ini_dir> <tiles_out_dir> <sheet1.tif> [...] [--zoom 6,14] [--workers 2]
"""
import argparse, glob, json, math, os, re, sys
from multiprocessing import Pool
import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
from pyproj import Transformer
from tiles_2kf_batch import composite_save, merge_bounds, R

CFG = dict(P=None, sheets=None, inis=None, covered_url=None, skip=())


def configure(outlay_dir, ini_dir, covered_url=None, skip=()):
    ini = open(os.path.join(outlay_dir, "geoview_mk.ini"), encoding="latin-1").read()
    CFG["P"] = {k: float(v) for k, v in re.findall(r"^(ENcorr_\w+)=([-\d.E+]+)", ini, flags=re.M)}
    xml = open(os.path.join(outlay_dir, "ekf_mk.xml"), encoding="latin-1").read()
    CFG["sheets"] = {}
    for gx, gy, fn, cx, cy, ox, oy in re.findall(
            r'<file gridx="(\d+)" gridy="(\d+)" filename="([^"]+)"[^>]*geo:cellx="([-\d.]+)" geo:celly="([-\d.]+)" geo:origx="([-\d.]+)" geo:origy="([-\d.]+)"', xml):
        CFG["sheets"][os.path.splitext(fn)[0].lower()] = dict(cx=float(cx), cy=float(cy), ox=float(ox), oy=float(oy))
    CFG["inis"] = {os.path.splitext(os.path.basename(p))[0].lower(): p
                   for p in glob.glob(os.path.join(ini_dir, "**", "*.ini"), recursive=True)}
    CFG["covered_url"] = covered_url
    CFG["skip"] = {x.lower() for x in skip}
    return len(CFG["sheets"]), len(CFG["inis"])


def grid_to_utm33(x, y):
    P = CFG["P"]
    x, y = x * P["ENcorr_tometer_X"], y * P["ENcorr_tometer_Y"]
    E = P["ENcorr_a00"] + P["ENcorr_a10"]*x + P["ENcorr_a01"]*y + P["ENcorr_a20"]*x*x + P["ENcorr_a11"]*x*y + P["ENcorr_a02"]*y*y
    N = P["ENcorr_b00"] + P["ENcorr_b10"]*x + P["ENcorr_b01"]*y + P["ENcorr_b20"]*x*x + P["ENcorr_b11"]*x*y + P["ENcorr_b02"]*y*y
    return E, N


def read_clip(sheet):
    p = CFG["inis"].get(sheet.lower())
    if not p:
        return None
    m = re.search(r"clip=(\d+),(\d+),(\d+),(\d+)", open(p, encoding="latin-1").read())
    if not m:
        return None
    x, y, w, h = map(int, m.groups())
    return x, y, x + w, y + h


def sheet_affine_3857(s, x0, y0, x1, y1):
    """Affine from the clipped window's pixels to EPSG:3857, fitted to a 5x5 grid of points mapped
    through the DVD polynomial (over one sheet the polynomial is almost linear)."""
    utm_to_merc = Transformer.from_crs("EPSG:32633", "EPSG:3857", always_xy=True)
    A, B = [], []
    for i in range(5):
        for j in range(5):
            px = x0 + (x1 - x0) * i / 4; py = y0 + (y1 - y0) * j / 4
            E, N = grid_to_utm33(s["ox"] + px * s["cx"], s["oy"] + py * s["cy"])
            A.append((px - x0, py - y0, 1)); B.append(utm_to_merc.transform(E, N))
    sol, *_ = np.linalg.lstsq(np.array(A), np.array(B), rcond=None)
    (a, d), (b, e), (c, f) = sol
    return Affine(a, b, c, d, e, f)


def already_covered(tf, W, H, z=12):
    """True if the existing layer really shows map at the sheet's centre and four quarter points.
    A tile that exists but is transparent at that point (outside the layer's own outline) does not count."""
    import io, requests
    from PIL import Image
    size = 2 * R / 2 ** z
    for fx, fy in ((.5, .5), (.25, .25), (.75, .25), (.25, .75), (.75, .75)):
        X, Y = tf * (W * fx, H * fy)
        tx, ty = int((X + R) // size), int((R - Y) // size)
        px = min(255, int(((X + R) / size - tx) * 256)); py = min(255, int(((R - Y) / size - ty) * 256))
        r = None
        for attempt in range(3):
            try:
                r = requests.get(CFG["covered_url"].format(z=z, x=tx, y=ty), timeout=20); break
            except Exception:
                r = None
        if r is None or r.status_code != 200:
            return False
        try:
            if Image.open(io.BytesIO(r.content)).convert("RGBA").getpixel((px, py))[3] < 128:
                return False
        except Exception:
            return False
    return True


def process(job):
    tif, out_dir, zmin, zmax, bbox = job
    sheet = os.path.splitext(os.path.basename(tif))[0]
    res = dict(sheet=sheet, status="", tiles=[], bounds={})
    try:
        if sheet.lower() in CFG["skip"]:
            res["status"] = "skipped: already placed by hand (control points)"; return res
        s = CFG["sheets"].get(sheet.lower())
        if not s:
            res["status"] = "skipped: not in ekf_mk.xml"; return res
        with rasterio.open(tif) as ds:
            clip = read_clip(sheet)
            x0, y0, x1, y1 = clip or (0, 0, ds.width, ds.height)
            x1, y1 = min(x1, ds.width), min(y1, ds.height)
            src_tf = sheet_affine_3857(s, x0, y0, x1, y1)
            W, H = x1 - x0, y1 - y0
            cs = [src_tf * p for p in ((0, 0), (W, 0), (0, H), (W, H))]
            minX, maxX = min(p[0] for p in cs), max(p[0] for p in cs); minY, maxY = min(p[1] for p in cs), max(p[1] for p in cs)
            if bbox:
                to_wgs = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
                lon0, lat0 = to_wgs.transform(minX, minY); lon1, lat1 = to_wgs.transform(maxX, maxY)
                if lon1 < bbox[0] or lon0 > bbox[2] or lat1 < bbox[1] or lat0 > bbox[3]:
                    res["status"] = "skipped: outside bbox"; return res
            if CFG["covered_url"] and already_covered(src_tf, W, H):
                res["status"] = "skipped: already in the existing layer"; return res
            data = ds.read([1, 2, 3], window=Window(x0, y0, W, H))
        res["status"] = "ok" if clip else "ok, not clipped (no .ini for this sheet)"
        rgba = np.zeros((4,) + data.shape[1:], np.uint8)
        rgba[:3] = data; rgba[3] = np.where(data.any(axis=0), 255, 0); del data
        lock_path = os.path.join(os.path.dirname(os.path.abspath(out_dir)), ".tiles.lock")
        prev = prev_tf = None
        for z in range(zmax, zmin - 1, -1):
            size = 2 * R / 2 ** z
            xa, xb = int((minX + R) // size), int((maxX + R) // size)
            ya, yb = int((R - maxY) // size), int((R - minY) // size)
            res["bounds"][z] = dict(xMin=xa, xMax=xb, yMin=ya, yMax=yb)
            mos = np.zeros((4, (yb - ya + 1) * 256, (xb - xa + 1) * 256), np.uint8)
            mtf = Affine(size / 256, 0, -R + xa * size, 0, -size / 256, R - ya * size)
            if prev is None:
                reproject(rgba, mos, src_transform=src_tf, src_crs="EPSG:3857", dst_transform=mtf, dst_crs="EPSG:3857",
                          resampling=Resampling.bilinear)
            else:
                reproject(prev, mos, src_transform=prev_tf, src_crs="EPSG:3857", dst_transform=mtf, dst_crs="EPSG:3857",
                          resampling=Resampling.average)
            prev, prev_tf = mos, mtf
            for x in range(xa, xb + 1):
                for y in range(ya, yb + 1):
                    dst = mos[:, (y - ya) * 256:(y - ya + 1) * 256, (x - xa) * 256:(x - xa + 1) * 256]
                    if dst[3].max() == 0:
                        continue
                    rel = f"{z}/{x}/{y}.png"
                    composite_save(os.path.join(out_dir, rel), np.ascontiguousarray(dst), lock_path)
                    res["tiles"].append(rel)
    except Exception as e:
        res["status"] = f"ERROR: {type(e).__name__}: {e}"
    return res


def run(tifs, out_dir, zmin=6, zmax=14, workers=2, bbox=None):
    """Process the given GeoTIFFs; returns the list of per-sheet result dicts."""
    assert CFG["sheets"], "call configure(outlay_dir, ini_dir) first"
    os.makedirs(out_dir, exist_ok=True)
    jobs = [(t, out_dir, zmin, zmax, bbox) for t in tifs]
    if workers > 1 and len(jobs) > 1:
        with Pool(workers) as p:
            results = p.map(process, jobs, chunksize=1)
    else:
        results = [process(j) for j in jobs]
    changed = sorted({t for r in results for t in r["tiles"]})
    with open(os.path.join(os.path.dirname(os.path.abspath(out_dir)), "changed_tiles.txt"), "w") as f:
        f.write("\n".join(changed) + ("\n" if changed else ""))
    return results


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("outlay_dir"); ap.add_argument("ini_dir"); ap.add_argument("out_dir"); ap.add_argument("tifs", nargs="+")
    ap.add_argument("--zoom", default="6,14"); ap.add_argument("--workers", type=int, default=2); ap.add_argument("--covered-url")
    a = ap.parse_args()
    print("sheets in ekf_mk.xml / .ini files found:", configure(a.outlay_dir, a.ini_dir, a.covered_url))
    zmin, zmax = map(int, a.zoom.split(","))
    total = {}
    for r in run(a.tifs, a.out_dir, zmin, zmax, a.workers):
        merge_bounds(total, r["bounds"])
        print(json.dumps({k: (len(v) if k == "tiles" else v) for k, v in r.items() if k != "bounds"}, ensure_ascii=False))
    print("tile bounds:", json.dumps(total))


In [ ]:
%%writefile /content/work/danfront_funcs.py
# ---- helper functions (pure Python; no Colab-specific calls) ----
import csv, json, math, os, shutil, subprocess, sys, time
import requests

def free_gb(path):
    return shutil.disk_usage(path).free / 1e9

def wait_for_disk(path, min_free=12, resume_at=20):
    """Pause while the local disk is nearly full (Drive needs time to sync its write cache)."""
    if free_gb(path) >= min_free:
        return
    print(f"  local disk low ({free_gb(path):.0f} GB free) - waiting for Drive to catch up ...")
    while free_gb(path) < resume_at:
        time.sleep(30)
    print("  continuing")

def download_parts(url, dest_prefix, part_size=4 * 1024**3, chunk=8 * 1024**2, disk_path="/content"):
    """Download url as raw volumes dest_prefix.001, .002, ... (7-Zip opens these directly).
    Finished parts are skipped, so the cell can be re-run with a fresh link after an interruption."""
    s = requests.Session()
    r = s.get(url, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
    cr = r.headers.get("Content-Range"); status = r.status_code; ctype = r.headers.get("Content-Type", ""); r.close()
    if status in (401, 403) or "text/html" in ctype:
        raise RuntimeError(f"The link was refused (HTTP {status}, {ctype}). It has probably expired: copy a fresh download link and run this cell again.")
    if status != 206 or not cr:
        raise RuntimeError(f"The server did not accept a ranged request (HTTP {status}). Send this message to Claude.")
    total = int(cr.split("/")[-1])
    size_file = dest_prefix + ".size"
    if os.path.exists(size_file):
        old = int(open(size_file).read().strip())
        if old != total:
            raise RuntimeError(f"This link points to a file of {total} bytes, but the parts already downloaded belong to a file of {old} bytes. Wrong link?")
    else:
        open(size_file, "w").write(str(total))
    nparts = math.ceil(total / part_size)
    print(f"archive size {total/1e9:.1f} GB -> {nparts} parts")
    t0 = time.time(); done_bytes = 0; new_bytes = 0
    for k in range(nparts):
        start = k * part_size; end = min(total, start + part_size) - 1; want = end - start + 1
        part = f"{dest_prefix}.{k+1:03d}"
        if os.path.exists(part) and os.path.getsize(part) == want:
            done_bytes += want; continue
        tmp = part + ".tmp"
        for attempt in range(5):
            try:
                r = s.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=120)
                if r.status_code != 206:
                    raise RuntimeError(f"HTTP {r.status_code}")
                got = 0
                with open(tmp, "wb") as f:
                    for c in r.iter_content(chunk):
                        f.write(c); got += len(c)
                        wait_for_disk(disk_path)
                if got != want:
                    raise IOError(f"got {got} of {want} bytes")
                os.replace(tmp, part); break
            except Exception as e:
                print(f"  part {k+1}: {e} - retrying ({attempt+1}/5)"); time.sleep(20)
        else:
            raise RuntimeError("Download keeps failing. The link has probably expired: copy a fresh download link and run this cell again (finished parts are kept).")
        done_bytes += want; new_bytes += want
        rate = new_bytes / max(1, time.time() - t0) / 1e6
        print(f"part {k+1}/{nparts} done  ({done_bytes/1e9:.1f} of {total/1e9:.1f} GB, {rate:.0f} MB/s)")
    have = sum(os.path.getsize(f"{dest_prefix}.{k+1:03d}") for k in range(nparts))
    assert have == total, f"size mismatch: {have} != {total}"
    print("download complete and verified:", f"{total/1e9:.1f} GB")

def list_archive(first_volume, exe="7z"):
    """Return (properties dict, [(path, size)] of GeoTIFFs with coordinates) from the archive listing."""
    out = subprocess.run([exe, "l", "-slt", first_volume], capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError("7z could not read the archive:\n" + out.stdout[-2000:] + out.stderr[-2000:])
    head, _, body = out.stdout.partition("----------")
    props = dict(l.split(" = ", 1) for l in head.splitlines() if " = " in l)
    sheets = []
    for block in body.split("\n\n"):
        e = dict(l.split(" = ", 1) for l in block.splitlines() if " = " in l)
        p = e.get("Path", "")
        if not p.lower().endswith((".tif", ".tiff")) or e.get("Folder") == "+" or e.get("Attributes", "").startswith("D"):
            continue
        if "/raw/" in "/" + p.replace("\\", "/"):
            continue                      # RAW sheets (no coordinates) are not for this notebook
        sheets.append((p, int(e.get("Size", 0))))
    return props, sorted(sheets)

def load_done(done_csv):
    if not os.path.exists(done_csv):
        return {}
    with open(done_csv, newline="", encoding="utf-8") as f:
        return {r["path"]: r for r in csv.DictReader(f)}

DONE_FIELDS = ["path", "sheet", "status", "cell_i", "cell_j", "cell_coverage", "tiles"]

def run_batches(cfg, max_sheets=None, batch_size=12, bbox=None, workers=2):
    """Extract -> tile -> upload -> record, batch by batch. Safe to interrupt and re-run."""
    import importlib
    tb = importlib.import_module(cfg.get("tiler", "tiles_2kf_batch"))   # tiles_1kf_batch for the First Survey
    done = load_done(cfg["done_csv"])
    todo = [p for p, _ in cfg["sheets"] if p not in done]
    if max_sheets:
        todo = todo[:max_sheets]
    print(f"{len(done)} sheets already done, {len(todo)} to do in this run")
    if (done or cfg.get("restore_always")) and not os.path.exists(cfg["restored_marker"]):
        print("new session: fetching the tiles made so far back from R2 (needed to merge sheet edges) ...")
        rc = subprocess.run(["rclone", "copy", cfg["remote"], cfg["tiles"], "--transfers", "64", "--checkers", "64"]).returncode
        if rc != 0:
            raise RuntimeError("could not restore tiles from R2")
    os.makedirs(cfg["tiles"], exist_ok=True); open(cfg["restored_marker"], "w").close()
    bounds = json.load(open(cfg["bounds_json"])) if os.path.exists(cfg["bounds_json"]) else {}
    bounds = {int(z): b for z, b in bounds.items()}
    t0 = time.time(); n_done = 0
    for b0 in range(0, len(todo), batch_size):
        batch = todo[b0:b0 + batch_size]
        wait_for_disk(cfg["disk_path"])
        shutil.rmtree(cfg["extract"], ignore_errors=True); os.makedirs(cfg["extract"])
        if cfg.get("fetch"):              # sheets come as single files (e.g. from a SharePoint folder)
            cfg["fetch"](batch, cfg["extract"])
        else:                             # sheets come out of the 7z archive
            lst = os.path.join(cfg["local"], "batch.txt")
            open(lst, "w", encoding="utf-8").write("\n".join(batch) + "\n")
            ex = subprocess.run([cfg.get("exe", "7z"), "x", cfg["first_volume"], f"-o{cfg['extract']}", f"-i@{lst}", "-y", "-bso0", "-bsp0"],
                                capture_output=True, text=True)
            if ex.returncode not in (0, 1):
                raise RuntimeError("7z extraction failed:\n" + ex.stdout[-1500:] + ex.stderr[-1500:])
        tifs, missing = [], []
        for p in batch:
            fp = os.path.join(cfg["extract"], p.replace("\\", "/"))
            (tifs if os.path.exists(fp) else missing).append((p, fp))
        results = tb.run([fp for _, fp in tifs], cfg["tiles"], 6, 14, workers=workers, bbox=bbox)
        changed = os.path.join(os.path.dirname(os.path.abspath(cfg["tiles"])), "changed_tiles.txt")
        if os.path.getsize(changed) > 0:
            for attempt in range(3):
                rc = subprocess.run(["rclone", "copy", cfg["tiles"], cfg["remote"], "--files-from", changed, "--no-traverse",
                                     "--transfers", "32"]).returncode
                if rc == 0:
                    break
                print("  upload failed, retrying ..."); time.sleep(15)
            else:
                raise RuntimeError("Upload to R2 failed 3 times. Check the R2 keys, then run this cell again.")
        rows = []
        for (p, _), r in zip(tifs, results):
            tb.merge_bounds(bounds, r["bounds"])
            c = r.get("cell", ["", ""])
            rows.append(dict(path=p, sheet=r["sheet"], status=r["status"], cell_i=c[0], cell_j=c[1],
                             cell_coverage=r.get("cell_coverage", ""), tiles=len(r["tiles"])))
        for p, _ in missing:
            rows.append(dict(path=p, sheet=os.path.basename(p), status="ERROR: not extracted", cell_i="", cell_j="", cell_coverage="", tiles=0))
        new_file = not os.path.exists(cfg["done_csv"])
        with open(cfg["done_csv"], "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=DONE_FIELDS)
            if new_file:
                w.writeheader()
            w.writerows(rows)
        json.dump(bounds, open(cfg["bounds_json"], "w"))
        shutil.rmtree(cfg["extract"], ignore_errors=True)
        n_done += len(batch)
        el = time.time() - t0; left = (len(todo) - n_done) * el / n_done
        bad = [r["sheet"] + ": " + r["status"] for r in rows if not r["status"].startswith(("ok", "skipped: already", "skipped: outside"))]
        print(f"[{n_done}/{len(todo)}] batch done in total {el/60:.0f} min, about {left/60:.0f} min left" + (f"   notes: {bad}" if bad else ""))
    print("run finished")

def stitch_preview(tiles_dir, z, max_px=1600):
    """Stitch all local tiles of zoom z into one image (for a quick look)."""
    import glob
    from PIL import Image
    tiles = glob.glob(os.path.join(tiles_dir, str(z), "*", "*.png"))
    if not tiles:
        return None
    key = lambda p: (int(p.split(os.sep)[-2]), int(os.path.basename(p)[:-4]))
    xs = sorted({key(p)[0] for p in tiles}); ys = sorted({key(p)[1] for p in tiles})
    W = Image.new("RGBA", (256 * (xs[-1] - xs[0] + 1), 256 * (ys[-1] - ys[0] + 1)), (225, 225, 225, 255))
    for p in tiles:
        x, y = key(p); W.alpha_composite(Image.open(p).convert("RGBA"), ((x - xs[0]) * 256, (y - ys[0]) * 256))
    W = W.convert("RGB")
    if max(W.size) > max_px:
        W.thumbnail((max_px, max_px))
    return W

def summary(cfg):
    done = load_done(cfg["done_csv"])
    from collections import Counter
    c = Counter(r["status"].split(":")[0] for r in done.values())
    print("sheets:", len(done), dict(c))
    for r in done.values():
        if (not r["status"].startswith(("ok", "skipped: already", "skipped: outside"))) or "irregular" in r["status"] or "not clipped" in r["status"]:
            print("  ", r["sheet"], "-", r["status"], "coverage", r["cell_coverage"])
    b = json.load(open(cfg["bounds_json"])) if os.path.exists(cfg["bounds_json"]) else {}
    print("\n--- copy everything below this line to Claude ---")
    print("sheets_ok:", sum(1 for r in done.values() if r["status"].startswith("ok")), "of", len(done))
    print(f"var {cfg.get('bounds_var', 'kf2TileBounds')} = {{")
    for z in sorted(b, key=int):
        v = b[z]; print(f"    {int(z)}: {{ xMin: {v['xMin']}, xMax: {v['xMax']}, yMin: {v['yMin']}, yMax: {v['yMax']} }},")
    print("};")

def repair(cfg):
    """After an interrupted run: fix half-written tiles and put failed sheets back on the to-do list."""
    import glob
    from PIL import Image
    bad = []
    for p in glob.glob(os.path.join(cfg["tiles"], "*", "*", "*.png")):
        try:
            with Image.open(p) as im:
                im.load()
        except Exception:
            bad.append(p)
    print(len(bad), "damaged tiles found")
    for p in bad:
        rel = os.path.relpath(p, cfg["tiles"]).replace(os.sep, "/")
        os.remove(p)
        subprocess.run(["rclone", "copyto", f"{cfg['remote']}/{rel}", p], capture_output=True)   # last good copy from R2, if any
        ok = False
        if os.path.exists(p):
            try:
                with Image.open(p) as im:
                    im.load()
                ok = True
            except Exception:
                os.remove(p)
        print("  ", rel, "- restored from R2" if ok else "- removed (will be rebuilt)")
    for t in glob.glob(os.path.join(cfg["tiles"], "*", "*", "*.tmp")):
        os.remove(t)
    if os.path.exists(cfg["done_csv"]):
        rows = list(csv.DictReader(open(cfg["done_csv"], newline="", encoding="utf-8")))
        keep = [r for r in rows if not r["status"].startswith("ERROR")]
        with open(cfg["done_csv"], "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=DONE_FIELDS); w.writeheader(); w.writerows(keep)
        print(len(rows) - len(keep), "failed sheets put back on the to-do list:", [r["sheet"] for r in rows if r["status"].startswith("ERROR")])


# ---- SharePoint folder access (for packages uploaded as single files, not as one archive) ----
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36"

class SharePointFolder:
    """Lists and downloads the files of a folder shared with an 'anyone with the link' sharing link."""
    def __init__(self, share_url):
        self.share_url = share_url
        self.login()

    def login(self):
        from urllib.parse import urlparse, parse_qs
        self.s = requests.Session(); self.s.headers["User-Agent"] = UA
        r = self.s.get(self.share_url, timeout=60)
        u = urlparse(r.url); q = parse_qs(u.query)
        if r.status_code != 200 or "id" not in q:
            raise RuntimeError(f"The sharing link did not open a folder (status {r.status_code}, landed on {r.url[:150]}). "
                               "Check that SHARE is the link of the folder itself.")
        self.root = q["id"][0]                                  # e.g. /personal/<user>/Documents/.../pc60_firstMS
        self.site = f"{u.scheme}://{u.netloc}" + "/".join(self.root.split("/")[:3])

    def _api(self, path, what):
        from urllib.parse import quote
        url = (f"{self.site}/_api/web/GetFolderByServerRelativePath(decodedurl='{quote(path.replace(chr(39), chr(39)*2))}')/{what}"
               "?$top=5000")
        for attempt in range(4):
            r = self.s.get(url, headers={"Accept": "application/json;odata=nometadata"}, timeout=60)
            if r.status_code == 200:
                return r.json()["value"]
            if r.status_code in (401, 403):
                self.login()
            time.sleep(3)
        raise RuntimeError(f"SharePoint did not list {path} (status {r.status_code}): {r.text[:300]}")

    def files(self, sub=""):
        """[(name, size, server_relative_url)] of the files directly in root/sub."""
        p = self.root + ("/" + sub.strip("/") if sub else "")
        return sorted((f["Name"], int(f["Length"]), f["ServerRelativeUrl"]) for f in self._api(p, "Files"))

    def folders(self, sub=""):
        p = self.root + ("/" + sub.strip("/") if sub else "")
        return sorted(f["Name"] for f in self._api(p, "Folders") if f["Name"] != "Forms")

    def download(self, rel_url, size, dest):
        from urllib.parse import quote
        if os.path.exists(dest) and os.path.getsize(dest) == size:
            return
        url = f"{self.site}/_layouts/15/download.aspx?SourceUrl={quote(rel_url)}"
        err = None
        for attempt in range(5):
            try:
                with self.s.get(url, stream=True, timeout=120) as r:
                    if r.status_code in (401, 403):
                        self.login(); raise IOError(f"status {r.status_code}")
                    r.raise_for_status()
                    with open(dest + ".part", "wb") as f:
                        for chunk in r.iter_content(4 * 1024**2):
                            f.write(chunk)
                if os.path.getsize(dest + ".part") != size:
                    raise IOError("size mismatch")
                os.replace(dest + ".part", dest); return
            except Exception as e:
                err = e; time.sleep(5 * (attempt + 1))
        raise RuntimeError(f"could not download {rel_url}: {err}")

    def download_many(self, items, dest_dir, threads=8):
        """items: [(name, size, rel_url)] -> dest_dir/name."""
        from concurrent.futures import ThreadPoolExecutor
        os.makedirs(dest_dir, exist_ok=True)
        with ThreadPoolExecutor(threads) as ex:
            list(ex.map(lambda it: self.download(it[2], it[1], os.path.join(dest_dir, it[0])), items))

    def download_tree(self, sub, dest_dir, threads=16):
        """Download root/sub with all its subfolders (for the small .ini files). Returns the number of files."""
        n = 0
        items = self.files(sub)
        self.download_many(items, dest_dir, threads); n += len(items)
        for d in self.folders(sub):
            n += self.download_tree(f"{sub}/{d}", os.path.join(dest_dir, d), threads)
        return n


In [ ]:
import sys, importlib
sys.path.insert(0, LOCAL)
import danfront_funcs as F, tiles_2kf_batch, tiles_1kf_batch as T
importlib.reload(tiles_2kf_batch); importlib.reload(F); importlib.reload(T)
print('scripts loaded')

## 3. Connect to R2
Reads the three secrets and checks that the bucket can be reached.

In [ ]:
os.environ.update({
    'RCLONE_CONFIG_R2_TYPE': 's3',
    'RCLONE_CONFIG_R2_PROVIDER': 'Cloudflare',
    'RCLONE_CONFIG_R2_ACCESS_KEY_ID': userdata.get('R2_ACCESS_KEY_ID').strip(),
    'RCLONE_CONFIG_R2_SECRET_ACCESS_KEY': userdata.get('R2_SECRET_ACCESS_KEY').strip(),
    'RCLONE_CONFIG_R2_ENDPOINT': userdata.get('R2_ENDPOINT').strip(),
    'RCLONE_S3_NO_CHECK_BUCKET': 'true',
})
REMOTE = 'r2:danfront/tiles-ext'
import subprocess
r = subprocess.run(['rclone', 'lsf', REMOTE + '/'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('R2 could not be reached. Check the three secrets (and that Notebook access is on).\n' + r.stderr[-800:])
print('R2 connection OK. Folders in tiles-ext:', r.stdout.split())

## 4. Open the SharePoint folder

`SHARE` is the sharing link of the `pc60_firstMS` folder (or of `DanFront_part8`). The cell lists the sheets, downloads the small
`.ini` files and loads the DVD georeference. If it fails, send the message to Claude.

In [ ]:
SHARE = "PASTE_THE_SHARING_LINK_OF_THE_FOLDER_HERE"
RAW_SUB = "1ms_Geotiff_export/raw"     # folder with the .tif files, relative to the shared folder
INI_SUB = "ini"                        # folder with the .ini files (and outlay/ekf_mk.xml, outlay/geoview_mk.ini)

SP = F.SharePointFolder(SHARE)
top = SP.folders()
print("shared folder:", SP.root.split("/")[-1], "| contains:", top)
if "pc60_firstMS" in top:              # the link points one level higher
    RAW_SUB, INI_SUB = "pc60_firstMS/" + RAW_SUB, "pc60_firstMS/" + INI_SUB
ITEMS = {n: (n, size, url) for n, size, url in SP.files(RAW_SUB) if n.lower().endswith((".tif", ".tiff"))}
SHEETS = sorted((n, it[1]) for n, it in ITEMS.items())
print(len(SHEETS), "sheets,", round(sum(s for _, s in SHEETS) / 1e9, 1), "GB | first ones:", [n for n, _ in SHEETS[:5]])

INI_DIR = f"{LOCAL}/ini"
print(SP.download_tree(INI_SUB, INI_DIR), ".ini and DVD files downloaded")
import glob
OUTLAY = os.path.dirname(glob.glob(f"{INI_DIR}/**/ekf_mk.xml", recursive=True)[0])

## 5. Settings
`COVERED` is the Arcanum layer that is already on the website: a sheet is skipped when that layer already shows it.
Set `COVERED = None` to tile every sheet. `SKIP` lists sheets that were placed by hand with control points and must not be replaced.

In [ ]:
COVERED = "https://danfront-tiles.odulsuzkisafilm.workers.dev/tiles/{z}/{x}/{y}.png"
SKIP = ["14-20"]
print("sheets in ekf_mk.xml / .ini files:", T.configure(OUTLAY, INI_DIR, covered_url=COVERED, skip=SKIP))

def fetch(batch, dest):                       # downloads one batch of sheets from SharePoint
    SP.download_many([ITEMS[n] for n in batch], dest)

CFG = dict(sheets=SHEETS, local=LOCAL, extract=f'{LOCAL}/sheets', tiles=f'{LOCAL}/tiles', disk_path='/content',
           done_csv=f'{WORK}/done.csv', bounds_json=f'{WORK}/tile_bounds.json', restored_marker=f'{LOCAL}/.restored',
           remote=REMOTE, tiler='tiles_1kf_batch', fetch=fetch, restore_always=True, bounds_var='extTileBounds')

# Ask the existing layer which sheets it already shows, before downloading anything (1-2 minutes).
from concurrent.futures import ThreadPoolExecutor
def covered(name):
    sheet = name.rsplit('.', 1)[0]
    s = T.CFG['sheets'].get(sheet.lower()); clip = T.read_clip(sheet)
    if not s or not clip:
        return None                           # not a sheet of the DVD grid, or no map frame known
    x0, y0, x1, y1 = clip
    return bool(COVERED) and T.already_covered(T.sheet_affine_3857(s, x0, y0, x1, y1), x1 - x0, y1 - y0)
names = [n for n, _ in SHEETS]
with ThreadPoolExecutor(16) as ex:
    cov = list(ex.map(covered, names))
missing = {n for n, c in zip(names, cov) if c is False}
print(len(names), 'sheets:', cov.count(True), 'already in the existing layer,', len(missing), 'to add,',
      cov.count(None), 'left out (no frame data):', [n for n, c in zip(names, cov) if c is None])
CFG['sheets'] = [(n, s) for n, s in SHEETS if n in missing]     # only these are downloaded and tiled
print(len(F.load_done(CFG['done_csv'])), 'sheets already processed')

## 6. Test with a few sheets (a few minutes)

Processes the first 24 of the sheets to add and shows them stitched together. If something looks wrong, stop here and send the picture and the messages to Claude.

In [ ]:
F.run_batches(CFG, max_sheets=24, batch_size=24, workers=os.cpu_count())
from IPython.display import display
display(F.stitch_preview(CFG['tiles'], 10))
import collections
print(collections.Counter(r['status'] for r in F.load_done(CFG['done_csv']).values()))

## 7. Process everything

Safe to interrupt and run again.

In [ ]:
BBOX = None        # e.g. (18.0, 44.0, 23.5, 47.0)
F.run_batches(CFG, batch_size=24, bbox=BBOX, workers=os.cpu_count())

## 8. Result
Overview picture and the summary. Copy the text below the line and send it to Claude: it is needed to update the layer in `maps.html`.

In [ ]:
display(F.stitch_preview(CFG['tiles'], 8))
F.summary(CFG)
print('tiled sheets:', sorted(r['sheet'] for r in F.load_done(CFG['done_csv']).values() if r['status'].startswith('ok')))

## 9. Repair (only if needed)

Run this if the summary lists sheets with `ERROR`, or after stopping cell 7 by hand. Then run cell 7 again, and cell 8.

In [ ]:
F.repair(CFG)